# Environment Setup & Toolchain Verification (TP 1 — Exercise 1)

## Assignment Objectives
1. **Validate the Python toolchain** required by the whole TAI track: `numpy`, `opencv-contrib-python` (with `cv2.ximgproc`), `matplotlib`, `scipy`, `scikit-image`, `pandas` and `scikit-learn`.
2. **Validate the unified repository layout**: the single consolidated, SHA-256 deduplicated image dataset `TAI/images/`, the shared `TAI/utils/` package, and the runtime-only `TAI/outputs/` folder.
3. Confirm that the environment works identically **locally (VS Code / PyCharm / JupyterLab / CLI nbconvert)** and **on Google Colab**.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

### What This Cell Does
- Detects Google Colab (`google.colab` in `sys.modules` or `COLAB_RELEASE_TAG`); on Colab it clones `APSD_TAI_LAB` into `/content` (unless already mounted from Drive), so the notebook works even when opened directly from GitHub.
- Locally, it walks upward from the current working directory to the repository root (marker: `README.md` + `APDS/` + `TAI/`) and registers it on `sys.path` — the notebook therefore runs identically from the repo root, from `TAI/…/notebooks/`, or from any sub-folder.
- Verifies `opencv-contrib-python` (checking `cv2.ximgproc`) and installs it only when missing.
- Calls `bootstrap()` from `TAI.utils.env_io`, which resolves `TAI/images/` (the single consolidated, SHA-256 deduplicated dataset) and `TAI/outputs/` (runtime artifacts, never committed).

### Why It Is Needed
- `cv2.imshow()` opens blocking GUI windows and freezes Jupyter kernels — everything here renders through inline Matplotlib instead.
- No hardcoded absolute paths (like `/home/...`): every path is derived from the discovered repository root, so the same cell works on any machine, in any IDE, and on Colab.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "TAI" / "utils").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
else:
    # Local IDE / Jupyter: walk upward to the repository root
    # (identified by README.md + APDS/ + TAI/), whatever the launch directory.
    _cwd = Path.cwd().resolve()
    _repo = next(
        (
            c
            for c in (_cwd, *_cwd.parents)
            if (c / "README.md").is_file() and (c / "APDS").is_dir() and (c / "TAI").is_dir()
        ),
        None,
    )
    if _repo is None:
        raise FileNotFoundError(
            f"APSD_TAI_LAB repository root not found above {_cwd}. "
            "Launch Jupyter from inside the cloned repository."
        )
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

# Make sure opencv-contrib-python is used (cv2.ximgproc) -- install only if missing
try:
    import cv2

    _has_ximgproc = hasattr(cv2, "ximgproc")
except ImportError:
    _has_ximgproc = False
if not _has_ximgproc:
    %pip install -q opencv-contrib-python
    import cv2

from TAI.utils.env_io import bootstrap, load_image, show_images, plot_image_with_hist

ROOT_DIR, IMG_DIR, OUT_DIR = bootstrap()

import math
import time

import matplotlib.pyplot as plt
import numpy as np

print(f"Repository root : {ROOT_DIR}")
print(f"Images folder   : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"Outputs folder  : {OUT_DIR}")
print(f"OpenCV          : {cv2.__version__} | cv2.ximgproc available: {hasattr(cv2, 'ximgproc')}")


---
## Toolchain Inventory

### What & Why
| Package | Role in the TAI labs |
|---|---|
| `numpy` | every from-scratch algorithm (vectorized slicing, histograms, morphology) |
| `opencv-contrib-python` | reference implementations + `cv2.ximgproc.thinning` (Zhang-Suen) |
| `matplotlib` | inline multi-panel visualizations (never `cv2.imshow`) |
| `scipy` / `scikit-image` | scientific helpers used by some analyses |
| `pandas` / `scikit-learn` | quantitative verification tables & APDS interoperability |

`opencv-contrib-python` (not bare `opencv-python`) is mandatory: only the contrib
build ships `cv2.ximgproc`.


In [ ]:
import importlib
import sys

print(f"Python : {sys.version.split()[0]}")
for pkg in ("numpy", "cv2", "matplotlib", "scipy", "skimage", "pandas", "sklearn"):
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"  [OK]      {pkg:12s} {ver}")
    except ImportError:
        status = "OPTIONAL" if pkg in ("scipy", "skimage") else "MISSING"
        print(f"  [{status:8s}] {pkg:12s}")

print(f"  OpenCV contrib (cv2.ximgproc): {hasattr(cv2, 'ximgproc')}")


---
## Unified Repository Layout & Shared Utilities

The whole TAI track shares **one** image dataset (`TAI/images/`) and **one**
Python package (`TAI/utils/`) so that no notebook duplicates path-discovery or
algorithm code. Generated artifacts always go to `TAI/outputs/` which is
git-ignored — committed images stay pristine.


In [ ]:
from TAI.utils import (
    add_gaussian_noise,
    bootstrap,
    calculate_psnr,
    calculate_ssim,
    find_repo_root,
    in_colab,
    load_image,
    show_images,
    sp_noise,
)

print(f"Repository root marker : {find_repo_root()}")
print(f"Running on Colab       : {in_colab()}")

images = sorted(p.name for p in IMG_DIR.iterdir() if p.is_file())
print(f"\nConsolidated dataset TAI/images ({len(images)} unique images):")
for name in images:
    print("  -", name)


In [ ]:
# End-to-end sanity check: load an image, corrupt it deterministically, measure PSNR/SSIM
im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)
im2 = add_gaussian_noise(im1, variance=100.0, seed=42)
im3 = sp_noise(im1, prob=0.10, seed=42)

print(f"im1 : shape={im1.shape}, dtype={im1.dtype}, range=[{im1.min()}, {im1.max()}]")
print(f"PSNR(im1, im2) Gaussian var=100 : {calculate_psnr(im1, im2):.2f} dB")
print(f"PSNR(im1, im3) S&P p=10%        : {calculate_psnr(im1, im3):.2f} dB")
print(f"SSIM(im1, im2)                  : {calculate_ssim(im1, im2):.4f}")
print(f"SSIM(im1, im1) identical        : {calculate_ssim(im1, im1):.4f}  (must be 1.0)")

show_images([
    ("im1 - Cameraman (grayscale)", im1),
    ("im2 - Gaussian noise", im2),
    ("im3 - Salt & pepper noise", im3),
], cols=3)


## Key Takeaways
- The repository ships **one** virtual environment (`.venv/` + root `requirements.txt`), **one** image dataset (`TAI/images/`) and **one** utilities package (`TAI/utils/`) shared by every TAI notebook.
- The bootstrap cell makes every notebook runnable from any working directory and on Google Colab without manual setup.
- With the toolchain validated above, all following labs (TP1 Exo2-5, TP2 Exo1-6) can be executed in order — or independently.
